# 02. Thử nghiệm & Tinh chỉnh Mô hình Random Forest (Người 3)

Notebook này tuân thủ **LUẬT CHUNG CỦA NHÓM**:
- Sử dụng `load_and_split_data(config)` để lấy đúng tập Train/Test theo tỉ lệ 80/20, `random_state=42`.
- Tinh chỉnh siêu tham số bằng `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` trên tập Train, `scoring='roc_auc'`.
- Kiểm tra hiện tượng Overfitting bằng cách so sánh `train_auc` và `test_auc`.
- Báo cáo đủ 5 chỉ số trên Test tại ngưỡng 0.5: `ROC-AUC`, `Gini`, `F1`, `Recall`, `Train-AUC`.
- Xuất đủ các file nộp vào `models/`, `reports/results/`, `reports/figures/`.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Thiết lập root path
ROOT_DIR = Path('..').resolve() if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

from src.utils.config import load_config
from src.data.loader import load_and_split_data
from src.modeling.model_rf import get_model, get_param_space, tune

print("Đã import thành công các module của dự án!")

## 1. Nạp Cấu hình & Dữ liệu

In [ ]:
config = load_config('configs/config.yaml')
X_train, X_test, y_train, y_test = load_and_split_data(config)

print(f"Tập Train: {X_train.shape}")
print(f"Tập Test : {X_test.shape}")
print("Tỷ lệ nợ xấu Train: ", round(y_train.mean() * 100, 2), "%")
print("Tỷ lệ nợ xấu Test : ", round(y_test.mean() * 100, 2), "%")

## 2. Kiểm tra Không gian Siêu tham số (get_param_space)

In [ ]:
param_space = get_param_space()
for k, v in param_space.items():
    print(f"{k}: {v}")

## 3. Tinh chỉnh Mô hình trên Tập Train (5-Fold Stratified CV, n_iter=40)

In [ ]:
# Chạy tune trên toàn bộ X_train
search = tune(config, X_train, y_train, n_iter=40)
print("CV ROC-AUC tốt nhất:", round(search.best_score_, 4))
print("Bộ tham số tối ưu:", search.best_params_)

## 4. Đánh giá Tập Test (Ngưỡng 0.5) & Kiểm tra Overfit

In [ ]:
from sklearn.metrics import roc_auc_score, f1_score, recall_score

best_pipe = search.best_estimator_

train_proba = best_pipe.predict_proba(X_train)[:, 1]
train_auc = float(roc_auc_score(y_train, train_proba))

test_proba = best_pipe.predict_proba(X_test)[:, 1]
test_auc = float(roc_auc_score(y_test, test_proba))
gini = 2 * test_auc - 1

test_pred_05 = (test_proba >= 0.5).astype(int)
f1 = float(f1_score(y_test, test_pred_05, pos_label=1))
recall = float(recall_score(y_test, test_pred_05, pos_label=1))

print("-" * 40)
print(f"Train ROC-AUC : {train_auc:.4f}")
print(f"Test ROC-AUC  : {test_auc:.4f}")
print(f"Chênh lệch AUC: {train_auc - test_auc:.4f}")
print(f"Gini Score    : {gini:.4f}")
print(f"F1 (Ngưỡng 0.5): {f1:.4f}")
print(f"Recall (Ngưỡng 0.5): {recall:.4f}")
print("-" * 40)